In [1]:
!pip install --upgrade sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.1/182.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
import pandas as pd
import numpy as np
path = '/content/drive/MyDrive/거래_이상탐지_프로젝트/data'
train_df = pd.read_csv(path+"/fraudTrain.csv")
test_df = pd.read_csv(path+"/fraudTest.csv")
print(train_df.shape, test_df.shape)

(1296675, 23) (555719, 23)


In [4]:
from sklearn.preprocessing import StandardScaler
import numpy as np
from bisect import bisect_left, bisect_right
from dateutil.relativedelta import relativedelta
import pandas as pd
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import roc_auc_score

def calculate_age(born, trans_date):
    return relativedelta(trans_date, born).years

def count_past_days_fast(df, days):
    results = []
    grouped = df.groupby('cc_num')

    for card, group in grouped:
        times = group['trans_date_trans_time'].tolist()
        counts = []
        for i, t in enumerate(times):
            start_time = t - pd.Timedelta(days=days)
            # 왼쪽 경계 (start_time 이상)
            left_idx = bisect_left(times, start_time)
            # 오른쪽 경계는 현재 거래 바로 전(i번째 거래 제외)
            right_idx = i
            counts.append(right_idx - left_idx)
        results.extend(counts)
    return results

def prep_df(df: pd.DataFrame) -> pd.DataFrame:
  # 원본 데이터프레임 복사
    df_p = df.copy()

    drop_cols = ['Unnamed: 0', 'unix_time', 'trans_num', 'first', 'last', 'merchant', 'street', 'merch_lat', 'merch_long', 'city_pop', 'lat', 'long', 'zip']
    df_p = df_p.drop(columns=drop_cols)

    # 로그 변환
    df_p['amt_log'] = np.log1p(df_p['amt'])

    # 표준화
    scaler = StandardScaler()
    df_p['amt_log_std'] = scaler.fit_transform(df_p[['amt_log']])

    # datetime 변환
    df_p['datetime'] = pd.to_datetime(df_p['trans_date_trans_time'])

    # 시(hour) 추출 (0~23)
    df_p['hour'] = df_p['datetime'].dt.hour

    # 요일 추출: Monday=0, Sunday=6
    df_p['day_of_week'] = df_p['datetime'].dt.dayofweek

    # 월 추출: 1월=1, 12월=12
    df_p['month'] = df_p['datetime'].dt.month

    df_p['trans_hour_sin'] = np.sin(2 * np.pi * df_p['hour'] / 24)
    df_p['trans_hour_cos'] = np.cos(2 * np.pi * df_p['hour'] / 24)

    # 금(4), 목(3), 수(2) + 심야 시간대(0~4, 21~23)
    high_risk_days = [2, 3, 4]  # 수, 목, 금
    high_risk_hours = list(range(0, 5)) + [21, 22, 23]

    df_p['is_high_risk_period'] = (
        df_p['day_of_week'].isin(high_risk_days) & df_p['hour'].isin(high_risk_hours)
    ).astype(int)

    df_p['trans_date_trans_time'] = pd.to_datetime(df_p['trans_date_trans_time'])
    df_p = df_p.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)

    df_p['cnt_1d'] = count_past_days_fast(df_p, 1)
    df_p['cnt_7d'] = count_past_days_fast(df_p, 7)
    df_p['cnt_30d'] = count_past_days_fast(df_p, 30)

    df_p['next_trans_time'] = df_p.groupby('cc_num')['trans_date_trans_time'].shift(-1)
    df_p['time_since_last_trans'] = (df_p['trans_date_trans_time'] - df_p.groupby('cc_num')['trans_date_trans_time'].shift(1)).dt.total_seconds()

    # 또는 마지막 거래 이후 경과 시간(다음 거래까지 시간 간격)
    df_p['time_until_next_trans'] = (df_p['next_trans_time'] - df_p['trans_date_trans_time']).dt.total_seconds()

    # 결측치 처리 (time_since_last_trans 첫 거래는 NaN, 0으로 채우거나 다른 방식)
    df_p[['time_since_last_trans', 'time_until_next_trans']] = df_p[['time_since_last_trans', 'time_until_next_trans']].fillna(0)


    df_p['dob'] = pd.to_datetime(df_p['dob'])

    df_p['age'] = df_p.apply(lambda x: calculate_age(x['dob'], x['trans_date_trans_time']), axis=1)
    df_p['age_group'] = (df_p['age'] // 10) * 10

    df_p['city_state'] = df_p['city'] + ', ' + df_p['state']

    # 최종 Drop
    drop_cols = ['trans_date_trans_time', 'dob', 'next_trans_time', 'cc_num', 'datetime', 'time_until_next_trans', 'city', 'state', 'amt', 'amt_log', 'hour']
    df_p = df_p.drop(columns=drop_cols, errors='ignore')

    return df_p

def preprocess_data(df: pd.DataFrame,
                    hour: str = 'sincos',
                    high_risk_period: bool = False,
                    age_group: bool = False
                    ) -> pd.DataFrame:
    '''
    Arg)
    df : 데이터프레임
    high_risk_period : high_risk_period 사용여부 (Default : False)
    age_group : age_group 사용여부 (Default : False)
    '''
    # 원본 데이터프레임 복사
    df_p = df.copy()

    if not high_risk_period:
        df_p = df_p.drop(columns=['is_high_risk_period'])

    # age_group 사용여부
    if age_group:
        df_p = df_p.drop(columns=['age'])
    else:
        df_p = df_p.drop(columns=['age_group'])

    return df_p

In [5]:
pre_train_df = preprocess_data(prep_df(train_df), 'sincos', True, False)
pre_test_df = preprocess_data(prep_df(test_df), 'sincos', True, False)

In [6]:
def m_estimate_smoothing(mean, global_mean, count, m):
    return (mean * count + global_mean * m) / (count + m)

In [7]:
# K-Fold 타겟 인코딩 함수
def kfold_target_encoding(df, target_col, cat_col, n_splits=5, m_param=10):
    df_encoded = df.copy()
    global_mean = df[target_col].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    # 컬럼명에 smoothing_type 대신 'm_est'와 m_param을 포함하여 명확하게
    oof_col = f'{cat_col}_enc_m_est_oof_m{m_param}'
    full_col = f'{cat_col}_enc_m_est_full_m{m_param}'
    df_encoded[oof_col] = np.nan

    # OOF 방식 인코딩 (교차검증용)
    for train_idx, valid_idx in kf.split(df):
        train_fold = df.iloc[train_idx]
        # agg는 훈련 폴드의 데이터로 집계
        agg = train_fold.groupby(cat_col)[target_col].agg(['mean', 'count']).reset_index()

        # m_estimate_smoothing 함수 사용
        agg['smoothed'] = m_estimate_smoothing(agg['mean'], global_mean, agg['count'], m_param)

        mapping = dict(zip(agg[cat_col], agg['smoothed']))
        # valid_fold의 해당 컬럼에 매핑하고, 매핑되지 않은 값(훈련 세트에 없던 범주)은 global_mean으로 채움
        df_encoded.loc[valid_idx, oof_col] = df.loc[valid_idx, cat_col].map(mapping).fillna(global_mean)

    # 전체 데이터 기준 인코딩 (최종 피처)
    agg_full = df.groupby(cat_col)[target_col].agg(['mean', 'count']).reset_index()
    # m_estimate_smoothing 함수 사용
    agg_full['smoothed'] = m_estimate_smoothing(agg_full['mean'], global_mean, agg_full['count'], m_param)
    full_mapping = dict(zip(agg_full[cat_col], agg_full['smoothed']))
    df_encoded[full_col] = df[cat_col].map(full_mapping).fillna(global_mean)

    return df_encoded, oof_col, full_col, full_mapping

In [8]:
# 1. GitHub에서 직접 클론
!git clone https://github.com/somepago/saint.git

# 2. 현재 세션에 해당 디렉토리를 패키지로 인식하도록 추가
import sys
sys.path.append('/content/saint')  # 또는 git clone한 경로

Cloning into 'saint'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 70 (delta 21), reused 19 (delta 19), pack-reused 35 (from 1)
Receiving objects: 100% (70/70), 209.36 KiB | 1.17 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [9]:
!pip install --upgrade sdv

In [10]:
!pip install openml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 61.7 MB/s eta 0:00:00
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=dad51650a1bd13f7e3c69e60b19db147ab2f3143ad1e0be876532cbd7a7155ba
  Stored in directory: /root/.cache/pip/wheels/00/23/31/5e562fce1f95aabe57f2a7320d07433ba1cd152bcde2f6a002
Successfully built liac-arff


In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from models import SAINT
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import numpy as np
from augmentations import embed_data_mask
from data_openml import DataSetCatCon


def evaluate_smoothing_with_saint_on_test_data(
    df_train, df_test, target_col, target_encode_cols, label_encode_cols, m_param,
    epochs=10, batch_size=512, device=None
):
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

    # Start with copies that will accumulate all processed features
    current_df_train = df_train.copy()
    current_df_test = df_test.copy()

    # --- 1. NaN 처리: 모든 관련 범주형 컬럼에 대해 'NA'로 채우기 ---
    all_categorical_features = list(set(target_encode_cols + label_encode_cols))
    for col in all_categorical_features:
        if col in current_df_train.columns:
            current_df_train[col] = current_df_train[col].fillna('NA')
        if col in current_df_test.columns:
            current_df_test[col] = current_df_test[col].fillna('NA')

    # --- 2. Target Encoding 적용 (target_encode_cols) ---
    target_encoded_feature_names = [] # _full_ 타겟 인코딩 컬럼의 이름을 저장
    global_mean_train_target = current_df_train[target_col].mean()

    for te_col in target_encode_cols:
        # kfold_target_encoding은 새 DataFrame을 반환하므로 할당해야 합니다.
        # df_encoded, oof_col, full_col, full_mapping 반환
        current_df_train, current_oof_col, current_full_col, current_full_mapping = kfold_target_encoding(
            current_df_train, target_col, te_col, n_splits=5, m_param=m_param
        )
        target_encoded_feature_names.append(current_full_col) # _full_ 컬럼 이름만 저장

        # 테스트 데이터에 동일한 매핑 적용
        current_df_test[current_full_col] = current_df_test[te_col].map(current_full_mapping).fillna(global_mean_train_target)
        # 테스트 데이터에는 OOF 컬럼을 추가하지 않습니다.

    # --- 3. Label Encoding 적용 (label_encode_cols) ---
    for le_col in label_encode_cols:
        # LabelEncoder 학습을 위한 모든 고유 범주 통합
        combined_le_categories = pd.concat([current_df_train[le_col], current_df_test[le_col]]).unique()
        le = LabelEncoder()
        le.fit(combined_le_categories)

        # Label Encoding 적용 (문자열로 변환하여 견고성 확보)
        current_df_train[le_col] = le.transform(current_df_train[le_col].astype(str))
        current_df_test[le_col] = le.transform(current_df_test[le_col].astype(str))

    # --- 4. SAINT 모델을 위한 최종 피처 컬럼 식별 ---
    # SAINT의 categorical input은 Label Encoding된 컬럼 (이미 정수)
    saint_categorical_cols = label_encode_cols

    # SAINT의 numerical input은 _full_ 타겟 인코딩된 컬럼과 원본 숫자형 컬럼
    saint_numerical_cols = []

    # 1. _full_ 타겟 인코딩 컬럼 추가
    saint_numerical_cols.extend(target_encoded_feature_names)

    # 2. 원본 DataFrame에서 숫자형 컬럼을 찾아 추가 (단, 이미 포함된 컬럼, 타겟 컬럼, 원본 범주형 컬럼 제외)
    # current_df_train의 컬럼을 기준으로 모든 숫자형 컬럼을 검토하되,
    # target_encode_cols나 label_encode_cols에 해당하는 원본 컬럼(이제는 변환되었거나 새 컬럼으로 대체됨)은 제외
    for col in current_df_train.columns:
        if col == target_col: # 타겟 컬럼은 제외
            continue
        if col in all_categorical_features: # 원본 범주형 컬럼 (이제는 인코딩되었으므로) 제외
            continue
        if pd.api.types.is_numeric_dtype(current_df_train[col]):
            # 이미 _full_ 타겟 인코딩 컬럼으로 추가된 경우 제외
            if col not in saint_numerical_cols:
                saint_numerical_cols.append(col)

    # 혹시 모를 경우를 대비한 OOF 컬럼 제외 (keyerror를 유발했던 원인 제거)
    saint_numerical_cols = [col for col in saint_numerical_cols if '_oof_m' not in col]

    # 디버깅을 위한 출력
    print(f"DEBUG: Final saint_categorical_cols: {saint_categorical_cols}")
    print(f"DEBUG: Final saint_numerical_cols: {saint_numerical_cols}")
    print(f"DEBUG: current_df_train.columns (after all processing): {current_df_train.columns.tolist()}")

    # 넘파이 배열로 변환하기 전, 실제로 DataFrame에 컬럼이 존재하는지 확인
    missing_cat_in_df = [c for c in saint_categorical_cols if c not in current_df_train.columns]
    missing_num_in_df = [c for c in saint_numerical_cols if c not in current_df_train.columns]

    if missing_cat_in_df or missing_num_in_df:
        error_msg = "KeyError: Some selected feature columns are missing from the processed DataFrame.\n"
        if missing_cat_in_df:
            error_msg += f"Missing categorical (Label Encoded) columns: {missing_cat_in_df}\n"
        if missing_num_in_df:
            error_msg += f"Missing numerical columns: {missing_num_in_df}\n"
        error_msg += "Please review data preprocessing and column selection logic."
        raise KeyError(error_msg)

    # DataFrame을 넘파이 배열로 변환
    X_cat_train = current_df_train[saint_categorical_cols].values.astype(np.int64) if saint_categorical_cols else np.array([]).reshape(-1, 0)
    X_num_train = current_df_train[saint_numerical_cols].values.astype(np.float32) if saint_numerical_cols else np.array([]).reshape(-1, 0)
    y_train = current_df_train[target_col].values.astype(np.int64)

    X_cat_test = current_df_test[saint_categorical_cols].values.astype(np.int64) if saint_categorical_cols else np.array([]).reshape(-1, 0)
    X_num_test = current_df_test[saint_numerical_cols].values.astype(np.float32) if saint_numerical_cols else np.array([]).reshape(-1, 0)
    y_test = current_df_test[target_col].values.astype(np.int64)

    # shape check for empty arrays for hstack
    if X_cat_train.shape[1] == 0 and X_num_train.shape[1] == 0:
        raise ValueError("No features identified for SAINT model after preprocessing.")
    if X_cat_train.shape[0] == 0 or X_num_train.shape[0] == 0:
        if X_cat_train.shape[0] == 0 and X_num_train.shape[0] > 0:
            X_cat_train = np.zeros((X_num_train.shape[0], 0), dtype=np.int64)
        elif X_num_train.shape[0] == 0 and X_cat_train.shape[0] > 0:
            X_num_train = np.zeros((X_cat_train.shape[0], 0), dtype=np.float32)

    if X_cat_test.shape[0] == 0 or X_num_test.shape[0] == 0:
        if X_cat_test.shape[0] == 0 and X_num_test.shape[0] > 0:
            X_cat_test = np.zeros((X_num_test.shape[0], 0), dtype=np.int64)
        elif X_num_test.shape[0] == 0 and X_cat_test.shape[0] > 0:
            X_num_test = np.zeros((X_cat_test.shape[0], 0), dtype=np.float32)

    # --- 5. 데이터 합치기 및 마스크 생성 ---
    X_train_combined = np.hstack([X_cat_train, X_num_train])
    X_test_combined = np.hstack([X_cat_test, X_num_test])

    mask_train = np.ones_like(X_train_combined, dtype=np.int64)
    mask_test = np.ones_like(X_test_combined, dtype=np.int64)

    # --- 6. cat_cols 위치 지정 (SAINT의 DataSetCatCon을 위한 인덱스) ---
    cat_cols_indices_for_dataset = list(range(X_cat_train.shape[1]))

    # 연속형 피처의 평균/표준편차는 학습 데이터에서 계산합니다.
    if X_num_train.shape[1] > 0:
        continuous_mean_std = [X_num_train.mean(axis=0), X_num_train.std(axis=0)]
    else:
        continuous_mean_std = [np.array([]), np.array([])]

    train_ds = DataSetCatCon(
        {'data': X_train_combined, 'mask': mask_train},
        {'data': y_train.reshape(-1, 1)},
        cat_cols_indices_for_dataset,
        task='clf',
        continuous_mean_std=continuous_mean_std
    )
    test_ds = DataSetCatCon(
        {'data': X_test_combined, 'mask': mask_test},
        {'data': y_test.reshape(-1, 1)},
        cat_cols_indices_for_dataset,
        task='clf',
        continuous_mean_std=continuous_mean_std
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # --- 7. SAINT 모델 초기화 ---
    cat_dims = [current_df_train[col].nunique() for col in saint_categorical_cols]
    model = SAINT(
        categories=tuple([1] + cat_dims) if cat_dims else (1,),
        num_continuous=X_num_train.shape[1],
        dim=32,
        dim_out=1,
        depth=1,
        heads=4,
        attn_dropout=0.1,
        ff_dropout=0.1,
        cont_embeddings='MLP',
        attentiontype='colrow',
        final_mlp_style='sep',
        y_dim=2
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)

    # --- 8. 모델 학습 ---
    print("--- 모델 학습 시작 ---")
    model.train()
    for ep in range(epochs):
        total_loss = 0
        for batch in train_loader:
            x_cat, x_cont, yb, cat_mask, con_mask = [b.to(device) for b in batch]
            _, x_cat_enc, x_cont_enc = embed_data_mask(x_cat, x_cont, cat_mask, con_mask, model, vision_dset=False)
            reps = model.transformer(x_cat_enc, x_cont_enc)
            y_out = model.mlpfory(reps[:, 0, :])
            loss = criterion(y_out, yb.squeeze())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {ep+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
    print("--- 모델 학습 완료 ---")

    # --- 9. 테스트 데이터로 평가 ---
    print("\n--- 테스트 데이터 평가 시작 ---")
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for batch in test_loader:
            x_cat, x_cont, yb, cat_mask, con_mask = [b.to(device) for b in batch]
            _, x_cat_enc, x_cont_enc = embed_data_mask(x_cat, x_cont, cat_mask, con_mask, model, vision_dset=False)
            reps = model.transformer(x_cat_enc, x_cont_enc)
            y_out = model.mlpfory(reps[:, 0, :])
            probs = torch.softmax(y_out, dim=1)[:, 1]
            all_preds.append(probs.cpu().numpy())
            all_trues.append(yb.cpu().numpy())

    preds = np.concatenate(all_preds).flatten()
    trues = np.concatenate(all_trues).flatten()

    auc = roc_auc_score(trues, preds)

    pred_labels = (preds >= 0.5).astype(int)
    print("\nConfusion Matrix:\n", confusion_matrix(trues, pred_labels))
    print("Precision:", precision_score(trues, pred_labels, zero_division=0))
    print("Recall:", recall_score(trues, pred_labels, zero_division=0))
    print("F1 Score:", f1_score(trues, pred_labels, zero_division=0))
    print("ROC AUC:", roc_auc_score(trues, preds))
    print("Classification Report:\n", classification_report(trues, pred_labels, zero_division=0))

    print(f"\n✅ 테스트 세트 SAINT ROC AUC: {auc:.4f}")
    return auc

# CTGAN 적용

In [12]:
from sdv.single_table.ctgan import CTGAN
from sklearn.preprocessing import LabelEncoder

def generate_ctgan_augmented_data(df, target_col='is_fraud', n_samples=55000, epochs=300):
    fraud_df = df[df[target_col] == 1].copy()
    cat_cols = fraud_df.select_dtypes(include=['object', 'category']).columns.tolist()

    label_encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        fraud_df[col] = le.fit_transform(fraud_df[col].astype(str))
        label_encoders[col] = le

    model = CTGAN(epochs=epochs)
    model.fit(fraud_df.drop(columns=[target_col]))

    samples = model.sample(n_samples)

    for col in cat_cols:
        samples[col] = samples[col].round().astype(int)
        samples[col] = samples[col].clip(0, len(label_encoders[col].classes_) - 1)
        samples[col] = label_encoders[col].inverse_transform(samples[col])

    samples[target_col] = 1
    return samples


In [15]:
# 타겟 컬럼 이름
target_col = 'is_fraud'
categorical_cols_for_te = ['job', 'city_state']

# 고정할 m-estimate 파라미터
fixed_m_param = 100

final_ctgan_smoothing_configs = {}

ctgan_samples = generate_ctgan_augmented_data(
    df=pre_train_df,
    target_col=target_col,
    n_samples=55000,
    epochs=300
)
augmented_df = pd.concat([pre_train_df, ctgan_samples], ignore_index=True)

In [20]:
augmented_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1351675 entries, 0 to 1351674
Data columns (total 16 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   category               1351675 non-null  object 
 1   gender                 1351675 non-null  object 
 2   job                    1351675 non-null  object 
 3   is_fraud               1351675 non-null  int64  
 4   amt_log_std            1351675 non-null  float64
 5   day_of_week            1351675 non-null  int32  
 6   month                  1351675 non-null  int32  
 7   trans_hour_sin         1351675 non-null  float64
 8   trans_hour_cos         1351675 non-null  float64
 9   is_high_risk_period    1351675 non-null  int64  
 10  cnt_1d                 1351675 non-null  int64  
 11  cnt_7d                 1351675 non-null  int64  
 12  cnt_30d                1351675 non-null  int64  
 13  time_since_last_trans  1351675 non-null  float64
 14  age               

In [25]:
for cat_col in categorical_cols_for_te:
    print(f"\n--- 범주형 컬럼 '{cat_col}'에 대해 m={fixed_m_param}으로 평가 ---")
    if cat_col not in augmented_df.columns or augmented_df[cat_col].dtype not in ['object', 'category']:
        print(f"⚠️ 경고: 컬럼 '{cat_col}'은(는 범주형 타입이 아니거나 없습니다. 건너뜁니다.")
        continue

    auc_score = evaluate_smoothing_with_saint_on_test_data (
        df_train=augmented_df,
        df_test=pre_test_df,
        target_col=target_col,
        target_encode_cols=categorical_cols_for_te,
        label_encode_cols=['category', 'gender'],
        m_param=fixed_m_param
    )

    final_ctgan_smoothing_configs[f"{cat_col}__CTGAN_M_Estimate"] = {
        "m_param": fixed_m_param,
        "best_auc": auc_score
    }

print("\n=== M-Estimate 스무딩 결과 ===")
for key, value in final_ctgan_smoothing_configs.items():
    print(f"{key}: {value}")


--- 범주형 컬럼 'job'에 대해 m=100으로 평가 ---
DEBUG: Final saint_categorical_cols: ['category', 'gender']
DEBUG: Final saint_numerical_cols: ['job_enc_m_est_full_m100', 'city_state_enc_m_est_full_m100', 'amt_log_std', 'day_of_week', 'month', 'trans_hour_sin', 'trans_hour_cos', 'is_high_risk_period', 'cnt_1d', 'cnt_7d', 'cnt_30d', 'time_since_last_trans', 'age']
DEBUG: current_df_train.columns (after all processing): ['category', 'gender', 'job', 'is_fraud', 'amt_log_std', 'day_of_week', 'month', 'trans_hour_sin', 'trans_hour_cos', 'is_high_risk_period', 'cnt_1d', 'cnt_7d', 'cnt_30d', 'time_since_last_trans', 'age', 'city_state', 'job_enc_m_est_oof_m100', 'job_enc_m_est_full_m100', 'city_state_enc_m_est_oof_m100', 'city_state_enc_m_est_full_m100']
--- 모델 학습 시작 ---
Epoch 1/10, Loss: 0.0219
Epoch 2/10, Loss: 0.0134
Epoch 3/10, Loss: 0.0123
Epoch 4/10, Loss: 0.0117
Epoch 5/10, Loss: 0.0106
Epoch 6/10, Loss: 0.0116
Epoch 7/10, Loss: 0.0098
Epoch 8/10, Loss: 0.0104
Epoch 9/10, Loss: 0.0092
Epoch 10/1

In [30]:
pre_test_df.nunique()

,0
category,14
gender,2
job,478
is_fraud,2
amt_log_std,37256
day_of_week,7
month,7
trans_hour_sin,22
trans_hour_cos,22
is_high_risk_period,2


In [29]:
pre_train_df.nunique()

,0
category,14
gender,2
job,494
is_fraud,2
amt_log_std,52928
day_of_week,7
month,12
trans_hour_sin,22
trans_hour_cos,22
is_high_risk_period,2
